# Lab 7: Serve your student

The capstone. You will take a distilled student through to a running service and measure it
against the teacher.

1. Train the student again from saved teacher scores, and save it as one folder.
2. Make an 8-bit copy and check what it cost.
3. Time three versions in the notebook.
4. Start the service and load-test it over the network.
5. Time the teacher, and compare accuracy and cost.
6. Choose a fallback threshold on validation data and score the cascade once on the test set.

This lab needs results from Labs 2 and 3. Allow about 45 minutes. Run the cells in order.

In [ ]:
from kd_course.env import check_environment
from kd_course.results import default_results_dir, require_result

report = check_environment()
print(report.summary())
assert report.ok, "Fix the problems listed above before continuing."

teacher_result = require_result("teacher_scored")
scores_dir = default_results_dir() / "teacher_scores"
assert (scores_dir / "pool.json").is_file(), "Run Lab 3 first: this lab uses its teacher scores."

SERVICE = "http://127.0.0.1:8100"
GPU_PRICE = 1.62  # example price per hour for the GPU machine; use your own

In [ ]:
from kd_course.data import load_banking77, make_splits
from kd_course.label_scoring import load_scores

bundle = load_banking77()
splits = make_splits(bundle.train, bundle.test)
LABELS = bundle.label_names
N = len(LABELS)
pool_scores = load_scores(scores_dir / "pool.json", splits.pool.texts)
gold_scores = load_scores(scores_dir / "gold_train.json", splits.gold_train.texts)

## 1. Train the student and save it

The same recipe as Lab 3. This time the result is written to disk.

In [ ]:
from kd_course.classifier import STUDENT_ENCODER_ID
from kd_course.distill import DistillSet, train_distilled_classifier
from kd_course.serving.export import save_classifier

train = DistillSet(
    texts=splits.gold_train.texts + splits.pool.texts,
    teacher_log_probs=gold_scores + pool_scores,
    labels=splits.gold_train.labels + [None] * len(splits.pool),
)
student = train_distilled_classifier(
    STUDENT_ENCODER_ID, train, splits.gold_val, num_labels=N, temperature=2.0, alpha=0.5
)
student_dir = default_results_dir() / "students" / "classifier"
save_classifier(student, LABELS, student_dir)
print("Saved to", student_dir)

## 2. An 8-bit copy

Size before and after, and a paired comparison of accuracy on the test set.

In [ ]:
from kd_course.metrics import accuracy, paired_bootstrap
from kd_course.serving.export import ClassifierPredictor, model_size_mb, quantize_int8

gpu = ClassifierPredictor(
    student.model, student.tokenizer, student.max_length, name="GPU, full precision"
)
small_model = quantize_int8(student.model)
cpu8 = ClassifierPredictor(small_model, student.tokenizer, student.max_length, name="CPU, 8-bit")

print(f"Full precision: {model_size_mb(student.model):,.0f} MB")
print(f"8-bit:          {model_size_mb(small_model):,.0f} MB")

full_pred = [label for label, _ in gpu.predict(splits.test.texts)]
small_pred = [label for label, _ in cpu8.predict(splits.test.texts)]
r = paired_bootstrap(accuracy, splits.test.labels, full_pred, small_pred)
print(
    f"Accuracy: {accuracy(splits.test.labels, full_pred):.1%} -> "
    f"{accuracy(splits.test.labels, small_pred):.1%}"
)
print(
    f"Change: {r.difference * 100:+.1f} points "
    f"(95% interval {r.low * 100:+.1f} to {r.high * 100:+.1f})"
)

## 3. Three versions, timed in the notebook

Batches of 32 messages, no network involved.

In [ ]:
import copy

from kd_course.serving.loadtest import cost_per_1k, run_load_test, summarize_latencies

cpu_model = copy.deepcopy(student.model).cpu().float()
cpu32 = ClassifierPredictor(
    cpu_model, student.tokenizer, student.max_length, name="CPU, full precision"
)

texts = splits.test.texts
batches = [texts[i : i + 32] for i in range(0, len(texts), 32)]
for predictor in (gpu, cpu32, cpu8):
    predictor.predict(batches[0])  # warm-up, not timed
    result = run_load_test(predictor.predict, batches)
    timing = summarize_latencies(result.latencies)
    print(
        f"{predictor.name:<22} {result.items_per_second:>8,.0f} messages/s   "
        f"median batch {timing['p50_ms']:.0f} ms"
    )

In [ ]:
import gc

import torch

val_results = gpu.predict(splits.gold_val.texts)  # kept for the fallback step
test_results = gpu.predict(splits.test.texts)

del gpu, cpu32, cpu8, cpu_model, small_model, student
gc.collect()
torch.cuda.empty_cache()

## 4. Start the service

Open a terminal on this machine, go to the repo folder, and run:

```
uv run --group labs kd-student-serve
```

Leave it running, then run the next cell. It waits up to a minute for the service to answer.

In [ ]:
import time

import httpx

client = httpx.Client(base_url=SERVICE, timeout=30)
for _ in range(60):
    try:
        if client.get("/health").status_code == 200:
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    raise SystemExit("The service is not answering on port 8100. Start it in a terminal first.")

print(client.get("/info").json()["model"], "is answering.")
print(client.post("/classify", json={"texts": texts[:2]}).json())

## 5. Load-test the service

First one message at a time, for latency. Then batches of 32 from four callers at once, for
throughput.

In [ ]:
def call_service(batch):
    response = client.post("/classify", json={"texts": list(batch)})
    response.raise_for_status()
    return response.json()


single = run_load_test(call_service, [[text] for text in texts[:300]])
print(
    "One message at a time:",
    {k: round(v, 1) for k, v in summarize_latencies(single.latencies).items()},
)

batched = run_load_test(call_service, batches * 3, concurrency=4)
print(
    f"Batches of 32, four callers: {batched.items_per_second:,.0f} messages/s, "
    f"errors {batched.errors}"
)
student_cost = cost_per_1k(GPU_PRICE, batched.items_per_second)
print(f"Student cost per 1,000 messages at full load: ${student_cost:.5f}")

## 6. The teacher, measured the same way

The teacher is loaded beside the running service; there is room for both. It also scores the
validation messages, which the fallback step needs.

In [ ]:
from kd_course.label_scoring import score_labels, score_or_load, top_labels
from kd_course.models import TEACHER_ID, load_causal_lm

teacher, teacher_tokenizer = load_causal_lm(TEACHER_ID)

sample = texts[:200]
start = time.perf_counter()
score_labels(teacher, teacher_tokenizer, sample, LABELS)
teacher_rate = len(sample) / (time.perf_counter() - start)
teacher_cost = cost_per_1k(GPU_PRICE, teacher_rate)

val_scores = score_or_load(
    scores_dir / "gold_val.json", teacher, teacher_tokenizer, splits.gold_val.texts, LABELS
)
val_teacher = top_labels(val_scores)
del teacher
gc.collect()
torch.cuda.empty_cache()

student_pred = [label for label, _ in test_results]
student_acc = accuracy(splits.test.labels, student_pred)
teacher_acc = teacher_result["metrics"]["accuracy"]
print(f"{'':<10}{'accuracy':>10}{'messages/s':>12}{'$ per 1,000':>14}")
print(f"{'Teacher':<10}{teacher_acc:>10.1%}{teacher_rate:>12,.0f}{teacher_cost:>14.5f}")
print(f"{'Student':<10}{student_acc:>10.1%}{batched.items_per_second:>12,.0f}{student_cost:>14.5f}")
ratio = teacher_cost / student_cost
print(f"The student is about {ratio:,.0f} times cheaper per message at full load.")

## 7. The teacher as a fallback

The threshold is chosen on the **validation** set: the lowest threshold whose combined accuracy
reaches the teacher's own validation accuracy. The cascade is then scored once on the test set.

In [ ]:
from kd_course.serving.cascade import cascade_stats, correct_flags, sweep_thresholds

val_conf = [confidence for _, confidence in val_results]
val_student_right = correct_flags([label for label, _ in val_results], splits.gold_val.labels)
val_teacher_right = correct_flags(val_teacher, splits.gold_val.labels)

thresholds = [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.01]
rows = sweep_thresholds(val_conf, val_student_right, val_teacher_right, thresholds)
print("On validation data")
for row in rows:
    print(
        f"  threshold {row['threshold']:<5} sent to teacher {row['sent_to_teacher']:>5.0%}   "
        f"accuracy {row['accuracy']:.1%}"
    )

target = rows[-1]["accuracy"]  # the teacher alone
chosen = next(row["threshold"] for row in rows if row["accuracy"] >= target)
print(f"Chosen threshold: {chosen}")

In [ ]:
test_conf = [confidence for _, confidence in test_results]
test_student_right = correct_flags(student_pred, splits.test.labels)
test_teacher_right = correct_flags(teacher_result["predictions"], splits.test.labels)

final = cascade_stats(test_conf, test_student_right, test_teacher_right, chosen)
print("On the test set, once:")
print(f"  Student alone  {student_acc:.1%}")
print(f"  Teacher alone  {teacher_acc:.1%}")
sent = final["sent_to_teacher"]
print(f"  Cascade        {final['accuracy']:.1%}, sending {sent:.0%} to the teacher")

## 8. What did you see?

Double-click this cell and write your answers. Then stop the service in its terminal with Ctrl+C.

**a. How much smaller was the 8-bit model? Did its accuracy really change? Quote the interval.**

**b. Would the 8-bit CPU version be fast enough for a chat product with a 50 ms target? For a nightly job over ten million messages?**

**c. Compare the service's single-message latency with the notebook's batch timing. Where does the difference come from?**

**d. How many times cheaper per message was the student? The figure assumes a fully busy machine. At 20,000 messages a day, what would the student really cost per 1,000?**

**e. What threshold did the fallback choose, and what share of traffic did it send to the teacher? What would you tell the person deciding whether to switch from the teacher to the student?**